In [1]:
import pandas as pd 
import numpy as np

In [34]:
df1 = pd.read_csv('cardio_alco.csv', sep=';')

In [35]:
df1.columns

Index(['id', 'alco'], dtype='object')

In [36]:
df1.head()

,id,alco
0,44,0
1,45,0
2,46,0
3,47,0
4,49,0


In [37]:
df1['id'].value_counts()

id
44       1
70423    1
70403    1
70407    1
70411    1
        ..
27170    1
27171    1
27172    1
27173    1
99954    1
Name: count, Length: 56903, dtype: int64

In [27]:
df1.columns

Index(['id;alco'], dtype='object')

In [10]:
df2 = pd.read_csv('cardio_base.csv')   

In [32]:
df2.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,smoke,age_years
0,0,18393,2,168,62.0,110,80,1,0,50
1,1,20228,1,156,85.0,140,90,3,0,55
2,2,18857,1,165,64.0,130,70,3,0,51
3,3,17623,2,169,82.0,150,100,1,0,48
4,4,17474,1,156,56.0,100,60,1,0,47


In [38]:
df2.columns = [c.strip() for c in df2.columns]
df1.columns = [c.strip() for c in df1.columns]

# Merge on 'id'
df = pd.merge(df1, df2, on="id", how="inner")

# Drop rows without alcohol info
df = df.dropna(subset=["alco"])

# Convert age from days to years (rounded down)
df["age_years"] = (df["age"] / 365).astype(int)

# Filter people over 50 years
over_50 = df[df["age_years"] > 50]

# Calculate percentage that consume alcohol (assuming 1 = drinks alcohol, 0 = does not)
percent_alcohol = (over_50["alco"].mean()) * 100

print(f"{percent_alcohol:.2f}% of people over 50 consume alcohol.")

4.91% of people over 50 consume alcohol.


In [39]:
from scipy.stats import ttest_ind

In [40]:
# ----------------------------
# 1️⃣ Smokers vs Non-Smokers
# ----------------------------
smokers = df2[df2["smoke"] == 1]
nonsmokers = df2[df2["smoke"] == 0]

# Compare cholesterol
t_chol, p_chol = ttest_ind(smokers["cholesterol"], nonsmokers["cholesterol"], equal_var=False)

# Compare systolic blood pressure (ap_hi)
t_bp, p_bp = ttest_ind(smokers["ap_hi"], nonsmokers["ap_hi"], equal_var=False)

# Compare weight
t_wt, p_wt = ttest_ind(smokers["weight"], nonsmokers["weight"], equal_var=False)

# ----------------------------
# 2️⃣ Men vs Women (Blood Pressure)
# ----------------------------
men = df2[df2["gender"] == 2]
women = df2[df2["gender"] == 1]
t_gender_bp, p_gender_bp = ttest_ind(men["ap_hi"], women["ap_hi"], equal_var=False)

# ----------------------------
# Report statistically significant differences (p < 0.05)
# ----------------------------
print("Smokers vs Non-Smokers")
print(f"Cholesterol: p = {p_chol:.5f}")
print(f"Blood Pressure: p = {p_bp:.5f}")
print(f"Weight: p = {p_wt:.5f}\n")

print("Men vs Women")
print(f"Blood Pressure: p = {p_gender_bp:.5f}")

Smokers vs Non-Smokers
Cholesterol: p = 0.00622
Blood Pressure: p = 0.46124
Weight: p = 0.00000

Men vs Women
Blood Pressure: p = 0.10384


In [15]:
df3 = pd.read_csv('covid_data.csv')

In [5]:
df1.columns

Index(['id;alco'], dtype='object')

In [11]:
df2.columns

Index(['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo',
       'cholesterol', 'smoke'],
      dtype='object')

In [12]:
df2.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,smoke
0,0,18393,2,168,62.0,110,80,1,0
1,1,20228,1,156,85.0,140,90,3,0
2,2,18857,1,165,64.0,130,70,3,0
3,3,17623,2,169,82.0,150,100,1,0
4,4,17474,1,156,56.0,100,60,1,0


In [13]:
# Clean column names (if needed)
df2.columns = [c.strip() for c in df2.columns]

# Convert age from days to years (rounded down)
df2["age_years"] = (df2["age"] / 365).astype(int)

# Group by age and calculate average weight
age_group_weight = df2.groupby("age_years")["weight"].mean()

# Find the heaviest and lightest average weights
max_weight = age_group_weight.max()
min_weight = age_group_weight.min()

# Compute the difference
difference = max_weight - min_weight

print(f"The heaviest age group is heavier by {difference:.2f} kg.")

The heaviest age group is heavier by 16.87 kg.


In [14]:
# Compute absolute and percentage difference
diff = max_weight - min_weight
percent_diff = (diff / min_weight) * 100

print(f"The heaviest age group is heavier by {diff:.2f} kg, which is {percent_diff:.2f}% heavier than the lightest age group.")

The heaviest age group is heavier by 16.87 kg, which is 28.60% heavier than the lightest age group.


In [17]:
# Split into two groups: over 50 and 50 or below
over_50 = df2[df2["age_years"] > 50]
under_50 = df2[df2["age_years"] <= 50]

# Compute average cholesterol levels
avg_over_50 = over_50["cholesterol"].mean()
avg_under_50 = under_50["cholesterol"].mean()

# Calculate percentage difference
percent_diff = ((avg_over_50 - avg_under_50) / avg_under_50) * 100

print(f"Average cholesterol (over 50): {avg_over_50:.2f}")
print(f"Average cholesterol (50 and below): {avg_under_50:.2f}")
print(f"People over 50 have {percent_diff:.2f}% higher cholesterol on average.")

Average cholesterol (over 50): 1.43
Average cholesterol (50 and below): 1.25
People over 50 have 14.69% higher cholesterol on average.


In [18]:
df2.columns = [c.strip() for c in df2.columns]

# Gender coding: usually 1 = female, 2 = male (verify if needed)
# Compute smoking rate per gender
smoking_rate = df2.groupby("gender")["smoke"].mean() * 100

# Extract male and female smoking rates
male_rate = smoking_rate.get(2, 0)
female_rate = smoking_rate.get(1, 0)

# Calculate percentage difference
percent_diff = male_rate - female_rate

print(f"Male smoking rate: {male_rate:.2f}%")
print(f"Female smoking rate: {female_rate:.2f}%")
print(f"Men are {percent_diff:.2f}% more likely to smoke than women.")

Male smoking rate: 21.89%
Female smoking rate: 1.79%
Men are 20.10% more likely to smoke than women.


In [19]:
# Find the height threshold for the tallest 1%
threshold = df2["height"].quantile(0.99)

print(f"The tallest 1% of people are taller than {threshold:.0f} cm.")

The tallest 1% of people are taller than 184 cm.


In [21]:
import pandas as pd

# Load df2 (cardio dataset)


# Compute Spearman correlation
corr = df2.corr(method='spearman')

# Find the top two correlated features (excluding self-correlation)
corr_unstacked = corr.unstack().sort_values(ascending=False)
corr_unstacked = corr_unstacked[corr_unstacked < 1]  # exclude self correlations

# Get the highest correlated pair
top_pair = corr_unstacked.idxmax()
top_value = corr_unstacked.max()

print(f"Highest Spearman correlation is between {top_pair[0]} and {top_pair[1]} = {top_value:.3f}")


Highest Spearman correlation is between age and age_years = 0.999


In [22]:
# Compute mean and standard deviation of height
mean_height = df2["height"].mean()
std_height = df2["height"].std()

# Identify people more than 2 std deviations from mean
outliers = df2[(df2["height"] > mean_height + 2 * std_height) | (df2["height"] < mean_height - 2 * std_height)]

# Calculate percentage
percentage = (len(outliers) / len(df2)) * 100

print(f"{percentage:.2f}% of people are more than 2 standard deviations away from the average height.")

3.34% of people are more than 2 standard deviations away from the average height.


In [16]:
df3.columns

Index(['location', 'date', 'new_cases', 'new_deaths', 'population',
       'aged_65_older_percent', 'gdp_per_capita',
       'hospital_beds_per_thousand'],
      dtype='object')

In [41]:
# Clean column names
df3.columns = [c.strip().lower() for c in df3.columns]

# Filter for Italy and Germany
italy = df3[df3["location"] == "Italy"].copy()
germany = df3[df3["location"] == "Germany"].copy()

# Ensure the 'date' column is datetime
italy["date"] = pd.to_datetime(italy["date"])
germany["date"] = pd.to_datetime(germany["date"])

# Compute cumulative total confirmed cases for each country
italy["total_cases"] = italy["new_cases"].cumsum()
germany["total_cases"] = germany["new_cases"].cumsum()

# Merge by date
merged = pd.merge(italy[["date", "total_cases"]],
                  germany[["date", "total_cases"]],
                  on="date",
                  suffixes=("_italy", "_germany"))

# Calculate absolute difference
merged["difference"] = (merged["total_cases_italy"] - merged["total_cases_germany"]).abs()

# Find the first date where difference exceeds 10,000
result = merged[merged["difference"] > 10000].head(1)

print(result[["date", "difference"]])

         date  difference
72 2020-03-12       10895


In [42]:
from scipy.optimize import curve_fit

In [43]:
# Filter for Italy and select the date range
italy = df3[df3["location"] == "Italy"].copy()
italy["date"] = pd.to_datetime(italy["date"])
italy = italy[(italy["date"] >= "2020-02-28") & (italy["date"] <= "2020-03-20")]

# Compute cumulative confirmed cases
italy["total_cases"] = italy["new_cases"].cumsum()

# Prepare x (days since start) and y (cases)
italy = italy.reset_index(drop=True)
x = np.arange(len(italy))  # 0, 1, 2, ..., N
y = italy["total_cases"].values

# Define exponential model: y = A * exp(Bx)
def exp_model(x, A, B):
    return A * np.exp(B * x)

# Fit the model to the data
params, _ = curve_fit(exp_model, x, y, p0=(y[0], 0.3))  # initial guess
A, B = params

# Predict for the last day (2020-03-20)
predicted = exp_model(x[-1], A, B)
actual = y[-1]
difference = abs(predicted - actual)

print(f"Predicted cases (model): {predicted:.0f}")
print(f"Actual cases: {actual}")
print(f"Difference: {difference:.0f}")

Predicted cases (model): 42347
Actual cases: 40635
Difference: 1712


In [44]:
# Calculate total deaths per country
death_per_country = df3.groupby("location", as_index=False).agg({
    "new_deaths": "sum",
    "population": "first"
})

# Compute death rate per million inhabitants
death_per_country["death_rate_per_million"] = (
    death_per_country["new_deaths"] / death_per_country["population"]
) * 1_000_000

# Sort descending and get 3rd highest
top3 = death_per_country.sort_values("death_rate_per_million", ascending=False).head(3)

print(top3)

       location  new_deaths  population  death_rate_per_million
163  San Marino          42     33938.0             1237.550828
18      Belgium        9619  11589616.0              829.967102
3       Andorra          51     77265.0              660.066007


In [46]:
from sklearn.metrics import f1_score
# Aggregate data per country
agg = df3.groupby("location", as_index=False).agg({
    "new_deaths": "sum",
    "population": "first",
    "aged_65_older_percent": "first"
})

# Remove missing data
agg = agg.dropna(subset=["population", "aged_65_older_percent", "new_deaths"])

# Compute death rate per million
agg["death_rate_per_million"] = (agg["new_deaths"] / agg["population"]) * 1_000_000

# Define true labels (actual condition)
# Countries that truly have death_rate > 50 per million
agg["actual"] = (agg["death_rate_per_million"] > 50).astype(int)

# Define predicted labels (based on rule: >20% population over 65 => expect high death rate)
agg["predicted"] = (agg["aged_65_older_percent"] > 20).astype(int)

# Compute F1 score
f1 = f1_score(agg["actual"], agg["predicted"])

print(f"F1 Score: {f1:.3f}")

F1 Score: 0.182


In [47]:
# Aggregate data by country
agg = df3.groupby("location", as_index=False).agg({
    "gdp_per_capita": "first",
    "hospital_beds_per_thousand": "first"
})

# Drop missing values
agg = agg.dropna(subset=["gdp_per_capita", "hospital_beds_per_thousand"])

# Filter for countries with at least 5 hospital beds per 1000 inhabitants
subset = agg[agg["hospital_beds_per_thousand"] >= 5]

# Compute conditional probability: P(GDP > 10000 | hospital_beds >= 5)
prob = (subset["gdp_per_capita"] > 10000).mean()

print(f"Probability: {prob:.2f}")

Probability: 0.88
